# Module 03: Sequence Modeling & Deep Learning for NLP
## Master Guide: Recurrent Neural Networks (RNN), LSTMs, and GRUs

Welcome to the Deep Learning phase of **NLP_101**! In Modules 1 & 2, you learned classical vectorization (Bag of Words, TF-IDF, Word2Vec). While powerful, those techniques suffer from three critical shortcomings:
1. **Bag of Words / TF-IDF discard word order completely**: "Dog bites man" and "Man bites dog" have identical representations.
2. **Fixed context window (N-grams)**: Exponentially explodes in dimensionality as $N$ increases.
3. **No state persistence**: Feed-forward networks process each word independently with no memory of what came before.

### 🎯 Learning Roadmap for this Notebook:
1. **The Sequential Paradigm & Recurrent Intuition**
2. **Standard Recurrent Neural Networks (Vanilla RNN)**
   - Mathematical formulation & hidden state recursion
   - Forward pass unrolled through time
   - Backpropagation Through Time (BPTT)
   - The vanishing & exploding gradient mathematical explanation
3. **Long Short-Term Memory Networks (LSTM)**
   - The Core Intuition: Cell State as the Information Highway
   - The 3 Gates: Forget Gate ($f_t$), Input Gate ($i_t$), Output Gate ($o_t$)
   - Complete step-by-step mathematical derivation
   - Why the additive cell state update eliminates the vanishing gradient problem
4. **Gated Recurrent Unit (GRU)**
   - Reset Gate ($r_t$) and Update Gate ($z_t$)
   - Comparison: RNN vs LSTM vs GRU (Parameters, compute, memory)
5. **Bidirectional & Multi-Layer Recurrent Models**
   - Forward and Backward hidden representations
6. **Hands-On Project in PyTorch**:
   - Building a production-grade Bidirectional LSTM Text Classifier from scratch
   - Embedding layer, LSTM layer, Dropout, Dense classification head
   - Training loop, loss curves, and live sentence prediction
7. **Hands-On Exercises & Self-Assessment Practice with Solutions**


---
## 1. Mathematical Foundations of Vanilla RNN

In a standard feedforward network, an input vector $x$ produces an output $\hat{y}$ through static weight matrices:
$$\hat{y} = g(W x + b)$$

In sequential data, our inputs arrive as an ordered sequence $(x_1, x_2, \dots, x_T)$. A Recurrent Neural Network introduces a recurrence relation by maintaining an internal **hidden state vector** $h_t \in \mathbb{R}^d$ that acts as memory:

$$h_t = 	anh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$
$$\hat{y}_t = 	ext{softmax}(W_{hy} h_t + b_y)$$

Where:
- $x_t \in \mathbb{R}^{d_{in}}$: Input word embedding at time step $t$
- $h_{t-1} \in \mathbb{R}^{d_h}$: Hidden state from previous time step (initialized to zeros $h_0 = \mathbf{0}$)
- $W_{xh} \in \mathbb{R}^{d_h 	imes d_{in}}$: Input-to-hidden weight matrix
- $W_{hh} \in \mathbb{R}^{d_h 	imes d_h}$: Hidden-to-hidden transition matrix (shared across ALL time steps!)
- $W_{hy} \in \mathbb{R}^{C 	imes d_h}$: Hidden-to-output projection matrix
- $	anh(\cdot)$: Activation function squashing activations to $(-1, +1)$

### Visualizing Unrolled RNN:
```text
      y_1          y_2          y_3                 y_T
       ^            ^            ^                   ^
       |            |            |                   |
    [W_hy]       [W_hy]       [W_hy]              [W_hy]
       |            |            |                   |
h_0 ->[RNN]--W_hh->[RNN]--W_hh->[RNN]-- ... -W_hh->[RNN] -> h_T
       ^            ^            ^                   ^
       |            |            |                   |
     [W_xh]       [W_xh]       [W_xh]              [W_xh]
       |            |            |                   |
      x_1          x_2          x_3                 x_T
```


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")


### 1.1 Building a Vanilla RNN Cell from Scratch in Pure NumPy
To truly master how RNNs work, let's implement the forward pass and hidden state updates manually without any deep learning framework.


In [ ]:
class ScratchVanillaRNNCell:
    """
    A pure NumPy implementation of a single Vanilla RNN step.
    Formula: h_t = tanh(W_hh @ h_prev + W_xh @ x_t + b_h)
    """
    def __init__(self, input_dim: int, hidden_dim: int):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        
        # Xavier/Glorot normal initialization
        scale_xh = np.sqrt(2.0 / (input_dim + hidden_dim))
        scale_hh = np.sqrt(2.0 / (hidden_dim + hidden_dim))
        
        self.W_xh = np.random.randn(hidden_dim, input_dim) * scale_xh
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * scale_hh
        self.b_h = np.zeros((hidden_dim, 1))
        
    def forward_step(self, x_t: np.ndarray, h_prev: np.ndarray) -> np.ndarray:
        """
        x_t: shape (input_dim, 1)
        h_prev: shape (hidden_dim, 1)
        returns h_t: shape (hidden_dim, 1)
        """
        linear_combo = np.dot(self.W_xh, x_t) + np.dot(self.W_hh, h_prev) + self.b_h
        h_t = np.tanh(linear_combo)
        return h_t
    
    def forward_sequence(self, sequence: np.ndarray) -> list:
        """
        Processes a sequence of tokens of shape (seq_len, input_dim)
        """
        seq_len = sequence.shape[0]
        h_t = np.zeros((self.hidden_dim, 1))
        hidden_states = []
        
        for t in range(seq_len):
            x_t = sequence[t].reshape(-1, 1)
            h_t = self.forward_step(x_t, h_t)
            hidden_states.append(h_t.copy())
            
        return hidden_states

# Test our scratch RNN cell
seq_length = 5
embed_dim = 16
hidden_size = 32

sample_seq = np.random.randn(seq_length, embed_dim)
rnn_cell = ScratchVanillaRNNCell(input_dim=embed_dim, hidden_dim=hidden_size)
all_hidden = rnn_cell.forward_sequence(sample_seq)

print(f"Input sequence length: {seq_length}")
print(f"Generated {len(all_hidden)} hidden states.")
print(f"Final hidden state shape: {all_hidden[-1].shape}")
print(f"Norm of final hidden state: {np.linalg.norm(all_hidden[-1]):.4f}")


---
## 2. Why Vanilla RNN Fails: The Vanishing and Exploding Gradient Problem

During Backpropagation Through Time (BPTT), the gradient of the loss at time step $T$ with respect to the hidden state at an early time step $t$ involves repeated matrix multiplications by $W_{hh}^T$:

$$\frac{\partial L_T}{\partial h_t} = \frac{\partial L_T}{\partial h_T} \prod_{k=t+1}^{T} \frac{\partial h_k}{\partial h_{k-1}}$$

Where:
$$\frac{\partial h_k}{\partial h_{k-1}} = \text{diag}(1 - \tanh^2(\dots)) \cdot W_{hh}^T$$

Notice two critical factors:
1. **The derivative of $\tanh(z)$ is bounded**: $\frac{d}{dz}\tanh(z) = 1 - \tanh^2(z) \in (0, 1]$. In fact, its maximum value is $1.0$ at $z=0$, and rapidly approaches $0$ as $|z|$ grows!
2. **Repeated multiplication of $W_{hh}^T$**:
   - If the largest eigenvalue $\lambda_{\max}(W_{hh}) < 1$, then $(W_{hh})^{T-t} \to 0$ exponentially fast as $(T - t)$ increases $\implies$ **Vanishing Gradient** (early words receive zero gradient update!).
   - If $\lambda_{\max}(W_{hh}) > 1$, $(W_{hh})^{T-t} \to \infty$ $\implies$ **Exploding Gradient** (NaNs, loss spikes).

Let's demonstrate this numerically with a 50-step sequence!


In [ ]:
# Simulation of Gradient Vanishing across 50 time steps
seq_steps = 50
W_hh_small = np.random.randn(10, 10) * 0.8  # Eigenvalues < 1
W_hh_large = np.random.randn(10, 10) * 1.5  # Eigenvalues > 1

grad_small = [1.0]
grad_large = [1.0]

current_small = np.eye(10)
current_large = np.eye(10)

for step in range(1, seq_steps):
    current_small = np.dot(current_small, W_hh_small * 0.9)  # tanh derivative approx
    current_large = np.dot(current_large, W_hh_large * 0.9)
    grad_small.append(np.linalg.norm(current_small))
    grad_large.append(np.linalg.norm(current_large))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(grad_small, color='crimson', lw=2)
plt.title("Vanishing Gradient: Norm shrinks to 0")
plt.xlabel("Time steps backwards (T - t)")
plt.ylabel("Gradient Norm")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(grad_large[:15], color='navy', lw=2)
plt.title("Exploding Gradient: Norm blows up")
plt.xlabel("Time steps backwards (T - t)")
plt.ylabel("Gradient Norm")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 3. Long Short-Term Memory (LSTM) Architecture

Proposed by Hochreiter & Schmidhuber (1997), the **LSTM** solves the vanishing gradient problem by introducing a dedicated **Cell State ($C_t$)** that acts as an unobstructed information conveyor belt.

Information flows along the cell state with only **linear** interactions (addition and element-wise scaling). The LSTM regulates what to read, write, and erase using **three gating mechanisms**:

```text
               +-------------------------------------------+
               |                                           |
               v                                           |
C_{t-1} -----[ (x) ]-----------------[ (+) ]-------------> C_t
               ^                       ^                   |
               | (f_t)                 | (i_t * ~C_t)      |
               |                       |                   |
            [sigma]        [sigma]   [tanh]     [sigma]  [tanh]
               ^              ^        ^           ^       |
               |              |        |           | (o_t) v
               +--------------+--------+           +----[ (x) ]---> h_t
               |                                           ^
       [x_t, h_{t-1}]                                      |
```

### The 4 Mathematical Operations of an LSTM:

1. **Forget Gate ($f_t$)**: Decides what fraction of past cell state to discard (0 = forget completely, 1 = retain completely).
   $$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$

2. **Input Gate ($i_t$) & Candidate State ($\tilde{C}_t$)**: Decides which new information to store in the cell state.
   $$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$
   $$\tilde{C}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)$$

3. **Cell State Update ($C_t$)**: Element-wise combination of filtered old memory and gated new candidate memory.
   $$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

4. **Output Gate ($o_t$) & Hidden State ($h_t$)**: Decides what part of the internal memory to expose as the output/hidden state.
   $$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$
   $$h_t = o_t \odot \tanh(C_t)$$

> 💡 **Why does LSTM prevent vanishing gradients?**  
> Notice the cell state update: $\frac{\partial C_t}{\partial C_{t-1}} = f_t + \dots$  
> If the forget gate $f_t \approx 1$, the gradient passes through time steps **unattenuated**! The gradient propagates via simple addition rather than repeated matrix multiplication by $W_{hh}^T$!


In [ ]:
class ScratchLSTMCell:
    """
    A complete NumPy implementation of an LSTM cell step.
    """
    def __init__(self, input_dim: int, hidden_dim: int):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        concat_dim = input_dim + hidden_dim
        
        # Combined weight matrix for [f, i, C_tilde, o]
        # Shape: (4 * hidden_dim, concat_dim)
        scale = np.sqrt(2.0 / concat_dim)
        self.W = np.random.randn(4 * hidden_dim, concat_dim) * scale
        self.b = np.zeros((4 * hidden_dim, 1))
        
        # Bias initialization trick: initialize forget gate bias to 1.0
        # This prevents the model from forgetting everything at the start of training!
        self.b[:hidden_dim] = 1.0

    @staticmethod
    def sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -15, 15)))

    def step(self, x_t: np.ndarray, h_prev: np.ndarray, C_prev: np.ndarray):
        """
        x_t: (input_dim, 1)
        h_prev: (hidden_dim, 1)
        C_prev: (hidden_dim, 1)
        returns h_t, C_t
        """
        # Concatenate hidden state and input
        combined = np.vstack((h_prev, x_t))  # (concat_dim, 1)
        
        # Linear projection
        gates = np.dot(self.W, combined) + self.b
        
        H = self.hidden_dim
        f_gate = self.sigmoid(gates[0*H : 1*H])
        i_gate = self.sigmoid(gates[1*H : 2*H])
        c_cand = np.tanh(gates[2*H : 3*H])
        o_gate = self.sigmoid(gates[3*H : 4*H])
        
        # Update cell state
        C_t = f_gate * C_prev + i_gate * c_cand
        
        # Compute output hidden state
        h_t = o_gate * np.tanh(C_t)
        
        return h_t, C_t

# Test Scratch LSTM Cell
lstm_test = ScratchLSTMCell(input_dim=8, hidden_dim=16)
x_sample = np.random.randn(8, 1)
h_init = np.zeros((16, 1))
C_init = np.zeros((16, 1))

h_1, C_1 = lstm_test.step(x_sample, h_init, C_init)
print("Scratch LSTM step executed successfully!")
print(f"Hidden state h_1 shape: {h_1.shape}, range: [{h_1.min():.3f}, {h_1.max():.3f}]")
print(f"Cell state C_1 shape: {C_1.shape}, range: [{C_1.min():.3f}, {C_1.max():.3f}]")


---
## 4. Gated Recurrent Unit (GRU)

Introduced by Cho et al. (2014), the **GRU** is a simplified, more computationally efficient variant of the LSTM.
- It **merges the cell state and hidden state** into a single vector $h_t$.
- It reduces the 3 gates of LSTM down to **2 gates**: **Reset Gate ($r_t$)** and **Update Gate ($z_t$)**.
- It has $\sim 25\%$ fewer parameters than an LSTM, making it faster to train and less prone to overfitting on smaller datasets.

### Mathematical Formulation:
1. **Reset Gate ($r_t$)**: Controls how much of previous hidden state to ignore when computing candidate memory.
   $$r_t = \sigma(W_r \cdot [h_{t-1}, x_t] + b_r)$$

2. **Update Gate ($z_t$)**: Acts like forget gate and input gate simultaneously!
   $$z_t = \sigma(W_z \cdot [h_{t-1}, x_t] + b_z)$$

3. **Candidate Hidden State ($\tilde{h}_t$)**:
   $$\tilde{h}_t = \tanh(W_h \cdot [r_t \odot h_{t-1}, x_t] + b_h)$$

4. **Hidden State Interpolation ($h_t$)**:
   $$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

### Architectural Comparison Table:
| Feature | Vanilla RNN | LSTM | GRU |
| :--- | :--- | :--- | :--- |
| **Memory Variables** | $h_t$ | $h_t$ and $C_t$ | $h_t$ |
| **Gating Mechanisms** | None | 3 (Forget, Input, Output) | 2 (Reset, Update) |
| **Parameter Count** | $O(d_h^2)$ | $4 \times O(d_h^2)$ | $3 \times O(d_h^2)$ |
| **Gradient Vanishing** | Severe | Solved via additive Cell State | Solved via additive Update gate |
| **Best Used For** | Educational study | Long documents, complex syntax | Smaller datasets, low latency |


---
## 5. Bidirectional RNNs (BiLSTM / BiGRU)

In standard sequential processing, $h_t$ only contains information from past tokens $(x_1, \dots, x_t)$. But in language understanding, context from the **future** is equally important!
Consider:
- *"I ate an **apple** for breakfast."* (Fruit)
- *"I bought an **Apple** laptop yesterday."* (Tech brand)

To disambiguate, a **Bidirectional Recurrent Model** runs two independent recurrent layers:
1. **Forward RNN ($\overrightarrow{h_t}$)**: Processes tokens from left to right: $x_1 \to x_2 \to \dots \to x_T$.
2. **Backward RNN ($\overleftarrow{h_t}$)**: Processes tokens from right to left: $x_T \to x_{T-1} \to \dots \to x_1$.

The final representation for each token is the concatenation:
$$h_t^{bi} = [\overrightarrow{h_t} ; \overleftarrow{h_t}] \in \mathbb{R}^{2 \cdot d_h}$$


---
## 6. End-to-End Hands-On Project: Bidirectional LSTM Text Classifier in PyTorch

Now, let's assemble everything we've learned into a real, production-ready deep learning text classification model!

We will build:
1. Custom vocabulary indexing & padding pipeline
2. PyTorch `Dataset` and `DataLoader`
3. Neural network with:
   - `nn.Embedding` (Trainable dense vector representations)
   - `nn.LSTM` with `bidirectional=True`, multi-layer stacking, and dropout
   - Linear classification head with LogSoftmax
4. Full training & evaluation loop with metrics
5. Real-time inference function for custom input sentences!


In [ ]:
import re
from collections import Counter
from torch.utils.data import Dataset, DataLoader

# 1. Synthetic dataset of customer sentiment reviews for classification
training_data = [
    ("This machine learning course is absolutely phenomenal and clear!", 1),
    ("I love the intuitive explanations and hands-on exercises.", 1),
    ("Brilliant tutorial, helped me understand deep learning easily.", 1),
    ("Outstanding guide! Highly recommend it to every data scientist.", 1),
    ("Fantastic content, the code runs smoothly without errors.", 1),
    ("Super helpful, clear math derivations and practical coding.", 1),
    ("Terrible quality, completely confusing and poorly explained.", 0),
    ("Waste of time, full of bugs and missing explanations.", 0),
    ("Awful experience, I could not understand anything from this lesson.", 0),
    ("Very disappointing, outdated materials and horrible code.", 0),
    ("Do not recommend this, it is frustrating and badly structured.", 0),
    ("Hopeless content, broken examples and confusing concepts.", 0),
]

# 2. Simple Tokenizer & Vocab Builder
def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

vocab = {"<PAD>": 0, "<UNK>": 1}
for sentence, _ in training_data:
    for word in tokenize(sentence):
        if word not in vocab:
            vocab[word] = len(vocab)

print(f"Vocabulary Size: {len(vocab)} unique tokens")
print(f"First 10 tokens: {list(vocab.items())[:10]}")


In [ ]:
# 3. Custom PyTorch Dataset
class TextClassificationDataset(Dataset):
    def __init__(self, data, vocab, max_len=15):
        self.data = data
        self.vocab = vocab
        self.max_len = max_len
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        text, label = self.data[idx]
        tokens = tokenize(text)
        
        # Convert tokens to indices with <UNK> fallback
        token_ids = [self.vocab.get(tok, self.vocab["<UNK>"]) for tok in tokens]
        
        # Pad or truncate to max_len
        if len(token_ids) < self.max_len:
            token_ids = token_ids + [self.vocab["<PAD>"]] * (self.max_len - len(token_ids))
        else:
            token_ids = token_ids[:self.max_len]
            
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

# Create DataLoader
dataset = TextClassificationDataset(training_data, vocab, max_len=12)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

sample_batch_x, sample_batch_y = next(iter(dataloader))
print(f"Batch X Shape: {sample_batch_x.shape} (batch_size, seq_len)")
print(f"Batch Y Shape: {sample_batch_y.shape} (batch_size,)")


In [ ]:
# 4. Define the Bidirectional LSTM Model Architecture
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers=2, dropout=0.3):
        super().__init__()
        
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )
        
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        # Bidirectional produces 2 * hidden_dim
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )
        
    def forward(self, x):
        # x: (batch_size, seq_len)
        embedded = self.embedding(x)  # (batch_size, seq_len, embed_dim)
        
        # output: (batch_size, seq_len, 2 * hidden_dim)
        # h_n: (num_layers * 2, batch_size, hidden_dim)
        output, (h_n, c_n) = self.lstm(embedded)
        
        # Concatenate final forward hidden state and final backward hidden state
        # Forward final: h_n[-2, :, :], Backward final: h_n[-1, :, :]
        final_forward = h_n[-2, :, :]
        final_backward = h_n[-1, :, :]
        combined_h = torch.cat((final_forward, final_backward), dim=1)  # (batch_size, 2 * hidden_dim)
        
        logits = self.fc(combined_h)
        return logits

# Initialize Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=32,
    hidden_dim=32,
    num_classes=2,
    num_layers=2,
    dropout=0.2
).to(device)

print(model)


In [ ]:
# 5. Training Loop
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

num_epochs = 35
losses = []

model.train()
for epoch in range(1, num_epochs + 1):
    epoch_loss = 0.0
    correct = 0
    total = 0
    
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        
        optimizer.step()
        
        epoch_loss += loss.item() * batch_x.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)
        
    avg_loss = epoch_loss / total
    accuracy = correct / total
    losses.append(avg_loss)
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/{num_epochs:02d} | Loss: {avg_loss:.4f} | Accuracy: {accuracy*100:.1f}%")

# Plot Loss Curve
plt.figure(figsize=(7, 3))
plt.plot(losses, color='dodgerblue', lw=2)
plt.title("BiLSTM Training Loss Convergence")
plt.xlabel("Epoch")
plt.ylabel("CrossEntropy Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# 6. Interactive Prediction Function for Any Arbitrary Text
def predict_sentiment(text, model, vocab, max_len=12):
    model.eval()
    with torch.no_grad():
        tokens = tokenize(text)
        token_ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
        if len(token_ids) < max_len:
            token_ids = token_ids + [vocab["<PAD>"]] * (max_len - len(token_ids))
        else:
            token_ids = token_ids[:max_len]
            
        input_tensor = torch.tensor([token_ids], dtype=torch.long).to(device)
        logits = model(input_tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        prediction = np.argmax(probs)
        
        label_map = {0: "NEGATIVE 🔴", 1: "POSITIVE 🟢"}
        return {
            "sentence": text,
            "prediction": label_map[prediction],
            "confidence": f"{probs[prediction] * 100:.2f}%",
            "probabilities": {"Negative": round(probs[0], 4), "Positive": round(probs[1], 4)}
        }

# Test predictions
test_sentences = [
    "This tutorial is really clear and wonderful!",
    "Terrible code, very frustrating and bad.",
    "I enjoyed understanding the math behind LSTMs.",
    "Disappointing lesson, waste of precious time."
]

print("=== LIVE INFERENCE DEMO ===")
for test_s in test_sentences:
    res = predict_sentiment(test_s, model, vocab)
    print(f"Text: '{res['sentence']}'")
    print(f"  -> Prediction: {res['prediction']} (Confidence: {res['confidence']})\n")


---
## 7. Hands-On Practice Exercises & Self-Assessment

Test your understanding of sequence models with these practical challenges!

### Exercise 1: Parameter Counting
Given an LSTM layer with `input_dim = 64` and `hidden_dim = 128`:
1. Calculate the exact number of trainable parameters in this single LSTM layer.
2. Compare it to a Vanilla RNN with the same dimensions.
*(Hint: Remember the 4 gates in LSTM, each with weights for inputs, hidden states, and biases).*

---
### Exercise 2: Implementing Dropout in Sequence Models
Why should you be careful when applying standard dropout across time steps inside an RNN/LSTM? What is "Variational Dropout" / "Recurrent Dropout"?

---
### Solutions:
<details>
<summary><b>Click to expand Solution 1</b></summary>

**For LSTM:**
An LSTM has 4 gates ($f, i, C, o$).
Each gate requires:
- Input weight: $W_x \in \mathbb{R}^{\text{hidden\_dim} \times \text{input\_dim}} = 128 \times 64 = 8,192$
- Hidden weight: $W_h \in \mathbb{R}^{\text{hidden\_dim} \times \text{hidden\_dim}} = 128 \times 128 = 16,384$
- Bias: $b \in \mathbb{R}^{\text{hidden\_dim}} = 128$
- Total per gate $= 8,192 + 16,384 + 128 = 24,704$
- Total for 4 gates $= 4 \times 24,704 = \mathbf{98,816}$ parameters!

**For Vanilla RNN:**
Only 1 set of weights:
- Total $= 8,192 + 16,384 + 128 = \mathbf{24,704}$ parameters.
*(LSTM has exactly $4\times$ the parameters of Vanilla RNN!)*
</details>

<details>
<summary><b>Click to expand Solution 2</b></summary>

Standard dropout randomly zeroes out different activations at every time step $t$. When applied recurrently across time steps, it disrupts the network's ability to maintain long-term memory across sequences.  
**Solution**: Variational / Recurrent Dropout samples a single dropout mask once and applies that **exact same binary mask** across all time steps for a given sequence.
</details>


---
## 8. Summary & Key Takeaways
1. **Vanilla RNNs** enable state persistence across sequences, but suffer from vanishing/exploding gradients during BPTT when sequence length exceeds 10-20 tokens.
2. **LSTMs** solve vanishing gradients by introducing a constant error carousel via an **additive Cell State** regulated by 3 gates (Forget, Input, Output).
3. **GRUs** compress the architecture into 2 gates (Reset, Update) with fewer parameters and faster training.
4. **Bidirectional models** capture both antecedent and subsequent context, critical for comprehensive semantic representations.
5. In **Module 04**, we will discover why even LSTMs hit a bottleneck when sequences span hundreds or thousands of tokens, leading directly to the invention of the **Attention Mechanism** and the **Transformer**!
